# Phase 1: EDA and Cleaning

Lending Club dataset exploration. Leave all work here — nothing in this notebook is the deliverable. Phase 1 cleaning logic lives in `pipeline/clean.py`.

Sections in order. Light instructions only; exploratory work is yours.

## Section 1: Load and Inspect

In [9]:
import pandas as pd
import numpy as np
from datetime import datetime

# Load raw data
df_a = pd.read_csv('/Users/gagepiercegaubert/Desktop/career_projects/lending-risk-intelligence/lending-risk-data/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv')
df_r = pd.read_csv('/Users/gagepiercegaubert/Desktop/career_projects/lending-risk-intelligence/lending-risk-data/rejected_2007_to_2018q4.csv/rejected_2007_to_2018Q4.csv')
df_a.head()

/var/folders/k8/0lnzlpcs1bn0sntqz4v2ryh40000gn/T/ipykernel_75360/2347681666.py:6: DtypeWarning: Columns (0,19,49,59,118,129,130,131,134,135,136,139,145,146,147) have mixed types. Specify dtype option on import or set low_memory=False.
  df_a = pd.read_csv('/Users/gagepiercegaubert/Desktop/career_projects/lending-risk-intelligence/lending-risk-data/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv')


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
df_a.shape

(2260701, 151)

In [11]:
with pd.option_context('display.max_rows', None):
    display(df_a.isna().sum().sort_values(ascending=False).to_frame('missing_count'))

,missing_count
member_id,2260701
orig_projected_additional_accrued_interest,2252050
hardship_end_date,2249784
hardship_start_date,2249784
hardship_type,2249784
hardship_reason,2249784
hardship_status,2249784
deferral_term,2249784
hardship_last_payment_amount,2249784
hardship_payoff_balance_amount,2249784


In [12]:
# Providing context of Missing Values against the total number of rows
total = len(df_a)
missing = df_a.isna().sum().sort_values(ascending=False)
missing_pct = (missing / total * 100).round(2)

pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})

,missing_count,missing_pct
member_id,2260701,100.00
orig_projected_additional_accrued_interest,2252050,99.62
hardship_end_date,2249784,99.52
hardship_start_date,2249784,99.52
hardship_type,2249784,99.52
...,...,...
policy_code,33,0.00
revol_bal,33,0.00
fico_range_high,33,0.00
fico_range_low,33,0.00


In [13]:
# Dropping all values with more than 50% missing
cols_to_drop = missing_pct[missing_pct > 50].index.tolist()
df_a = df_a.drop(columns=cols_to_drop)
df_a.shape

(2260701, 107)

## Section 2: Outcome Engineering — Binary Default

Explore `loan_status` values. Identify exact strings for Charged-Off, Default, Fully Paid, Current, Late, Grace Period.

In [14]:
# Exploring Loan Statuses
df_a['loan_status'].value_counts()

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

In [ ]:
terminal_states = [
    'Fully Paid',
    'Charged Off',
    'Default',
    'Does not meet the credit policy. Status:Fully Paid',
    'Does not meet the credit policy. Status:Charged Off',
]

default_states = [
    'Charged Off',
    'Default',
    'Does not meet the credit policy. Status:Charged Off',
]

df_clean = df_a[df_a['loan_status'].isin(terminal_states)].copy()
df_clean['is_default'] = df_clean['loan_status'].isin(default_states).astype(int)

pd.DataFrame({ 'metric': ['Rows before filter',
                          'Rows after filter',
                          'Dropped (non-terminal)',
                          'Default rate'], 'value':
                              [ f"{len(df_a):,}",
                               f"{len(df_clean):,}",
                               f"{len(df_a) - len(df_clean):,}",
                               f"{df_clean['is_default'].mean():.2%}", ] })

,metric,value
0,Rows before filter,"2,260,701"
1,Rows after filter,"1,348,099"
2,Dropped (non-terminal),"912,602"
3,Default rate,19.98%


## Section 3: Crisis Period Identification

Parse issue_d, identify 2008–2010 cohort, compare default rates.

In [16]:
# TODO: Parse issue_d and create crisis_period flag

df_clean['issue_d'] = pd.to_datetime(df_clean['issue_d'], format='%Y-%m-%d')

df_clean['issue_year'] = df_clean['issue_d'].dt.year

df_clean['crisis_period'] =  df_clean['issue_year'].between(2008, 2010).astype(int)

# Compare default rates: crisis vs. normal period

df_clean.groupby('crisis_period')['is_default'].mean().to_frame('default_rate')

ValueError: time data "Dec-2015" doesn't match format "%Y-%m-%d", at position 0. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

**Note:** Crisis period (2008–2010) shows a lower aggregate default rate than non-crisis, which looks backwards. This is survivorship bias — post-2015 "Current" loans were filtered out as non-terminal, leaving only resolved newer loans in the non-crisis group, which happened to default less. The flag is correct and retained for segmentation; the aggregate comparison just isn't meaningful here.

## Section 4: Null Analysis

Identify columns >50% null. Flag exceptions if column maps to a business segment.

In [ ]:
# Null Rates
null_rates = (df_a.isnull().sum() / len(df_a)).sort_values(ascending=False)
print("Null rates (top 20):")
print(null_rates.head(20))

Null rates (top 20):
il_util                  0.472810
mths_since_rcnt_il       0.402511
all_util                 0.383236
inq_last_12m             0.383139
total_cu_tl              0.383139
open_acc_6m              0.383139
open_rv_12m              0.383139
open_act_il              0.383139
open_il_12m              0.383139
total_bal_il             0.383139
open_il_24m              0.383139
open_rv_24m              0.383139
max_bal_bc               0.383139
inq_fi                   0.383139
mths_since_recent_inq    0.130698
emp_title                0.073872
num_tl_120dpd_2m         0.067983
emp_length               0.064998
mo_sin_old_il_acct       0.061531
bc_util                  0.033664
dtype: float64


## Section 5: Noise Removal

Identify columns with no signal: emp_title (150k categories), urls, identifiers, etc.

In [ ]:
# TODO: Inspect high-cardinality columns
# emp_title unique values?
# Other junk columns (url, id, member_id)?

high_cardinality = (
	df_clean.select_dtypes(include='object')
		.nunique(dropna=True)
		.sort_values(ascending=False)
)
display(high_cardinality.head(20))

title                   63151
zip_code                  945
earliest_cr_line          739
last_credit_pull_d        141
issue_d                   139
last_pymnt_d              136
addr_state                 51
sub_grade                  35
purpose                    14
emp_length                 11
grade                       7
home_ownership              6
loan_status                 5
verification_status         3
disbursement_method         2
application_type            2
term                        2
initial_list_status         2
debt_settlement_flag        2
hardship_flag               1
dtype: int64

In [ ]:
# Dropping emp_title since cardinality makres prediction difficult
noise_to_drop = [c for c in ['id', 'member_id', 'url', 'emp_title'] if c in df_clean.columns]
df_clean = df_clean.drop(columns=noise_to_drop)

## Section 6: Required Columns

For Phase 2 segmentation, which columns must be present (no nulls allowed)?

In [ ]:
# TODO: Check nulls in likely required columns
# loan_amnt, int_rate, term, grade, purpose, home_ownership, emp_length, issue_d
required = df_clean[[c for c in ['loan_amnt', 'int_rate', 'term', 'grade', 'purpose', 'home_ownership', 'emp_length', 'issue_d'] if c in df_clean.columns]]
required.isnull().sum()

loan_amnt             0
int_rate              0
term                  0
grade                 0
purpose               0
home_ownership        0
emp_length        78550
issue_d               0
dtype: int64

In [ ]:
rows_before = len(df_clean)
df_clean = df_clean.dropna(subset=['loan_amnt', 'int_rate', 'term', 'grade', 'purpose', 'home_ownership', 'emp_length', 'issue_d'])

## Section 7: Data Type and Format Inspection

Dates, percentages, categoricals. Any parsing needed?

In [ ]:
# Re-fromatting int_rate and revol_util to decimals, and earliest_cr_line to datetime
df_clean['int_rate'] = df_clean['int_rate'].astype(float) / 100
df_clean['revol_util'] = df_clean['revol_util'].astype(float) / 100

df_clean['earliest_cr_line'] = pd.to_datetime(df_clean['earliest_cr_line'], format='%b-%Y')

In [33]:
# Checking for inconsistencies in categorical variables
df_clean['purpose'].unique()

array(['debt_consolidation', 'small_business', 'home_improvement',
       'major_purchase', 'credit_card', 'other', 'house', 'vacation',
       'car', 'medical', 'moving', 'renewable_energy', 'wedding',
       'educational'], dtype=object)

In [35]:
# Cont.
df_clean['home_ownership'].unique()

array(['MORTGAGE', 'RENT', 'OWN', 'ANY', 'NONE', 'OTHER'], dtype=object)

In [37]:
df_clean['grade'].unique()

array(['C', 'B', 'F', 'A', 'E', 'D', 'G'], dtype=object)

## Section 8: Segmentation Preview

Quick look at default rates by grade, term, purpose before Phase 2.

In [41]:
# Segmenting values that signal a connection to default risk
df_clean.groupby('grade')['is_default'].mean().sort_index()

grade
A    0.060435
B    0.133963
C    0.224431
D    0.303803
E    0.384311
F    0.451464
G    0.496676
Name: is_default, dtype: float64

In [42]:
# A quick segmentation agg for grade and is_default
df_clean.groupby('grade')['is_default'].agg(
    loan_count='count',
    default_count='sum',
    default_rate='mean'
).sort_index()

,loan_count,default_count,default_rate
grade,,,
A,235193,14214,0.060435
B,393102,52661,0.133963
C,382323,85805,0.224431
D,201657,61264,0.303803
E,94192,36199,0.384311
F,32306,14585,0.451464
G,9326,4632,0.496676


In [43]:
# A quick seg agg for term and is_default
df_clean.groupby('term')['is_default'].agg(
    loan_count='count',
    default_count='sum',
    default_rate='mean'
).sort_index()

,loan_count,default_count,default_rate
term,,,
36 months,1023206,163926,0.160208
60 months,324893,105434,0.324519


**Note:** It does reflect correctly that the default rate increases as the grade goes down from A to G, and that 60-month loans have a higher default rate than 36-month loans. This is consistent with expectations, as lower grades and longer terms typically indicate higher risk.

## Section 9: Summary for clean.py

Before moving to Phase 1 pipeline, document your decisions here.

In [44]:
decisions = {
    'terminal_states': [
        'Fully Paid',
        'Charged Off',
        'Default',
        'Does not meet the credit policy. Status:Fully Paid',
        'Does not meet the credit policy. Status:Charged Off',
    ],
    'default_states': [
        'Charged Off',
        'Default',
        'Does not meet the credit policy. Status:Charged Off',
    ],
    'rows_before_filter': len(df_a),
    'rows_after_filter': len(df_clean),
    'default_rate': round(df_clean['is_default'].mean(), 4),
    'cols_dropped_null_threshold': '>50%',
    'cols_dropped_null': cols_to_drop,
    'noise_cols_dropped': ['id', 'member_id', 'url', 'emp_title'],
    'required_cols': ['loan_amnt', 'int_rate', 'term', 'grade', 'purpose', 'home_ownership', 'emp_length', 'issue_d'],
    'int_rate_conversion': 'divided by 100 (stored as float, no % sign)',
    'revol_util_conversion': 'divided by 100 (stored as float, no % sign)',
    'crisis_period_note': 'Survivorship bias — crisis period shows lower default rate than expected; flag retained for segmentation',
}

for k, v in decisions.items():
    print(f"{k}: {v}")

terminal_states: ['Fully Paid', 'Charged Off', 'Default', 'Does not meet the credit policy. Status:Fully Paid', 'Does not meet the credit policy. Status:Charged Off']
default_states: ['Charged Off', 'Default', 'Does not meet the credit policy. Status:Charged Off']
rows_before_filter: 2260701
rows_after_filter: 1348099
default_rate: 0.1998
cols_dropped_null_threshold: >50%
cols_dropped_null: ['member_id', 'orig_projected_additional_accrued_interest', 'hardship_end_date', 'hardship_start_date', 'hardship_type', 'hardship_reason', 'hardship_status', 'deferral_term', 'hardship_last_payment_amount', 'hardship_payoff_balance_amount', 'hardship_loan_status', 'hardship_dpd', 'hardship_length', 'payment_plan_start_date', 'hardship_amount', 'settlement_term', 'debt_settlement_flag_date', 'settlement_status', 'settlement_date', 'settlement_amount', 'settlement_percentage', 'sec_app_mths_since_last_major_derog', 'sec_app_revol_util', 'revol_bal_joint', 'sec_app_mort_acc', 'sec_app_fico_range_low',